# 012 Subgraphs

这是 LangGraph 学习线的第十二份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langgraph/use-subgraphs

学习目标：

1. 理解 subgraph 是把一个 graph 当作另一个 graph 的节点
2. 学会在父子 state schema 不同时用 wrapper node 调用子图
3. 学会在父子共享 state key 时把 compiled subgraph 直接 add_node
4. 学会用 `subgraphs=True` 观察子图 stream namespace
5. 理解子图 checkpointer 的 per-invocation / per-thread / stateless 模式
6. 跑通子图里的 interrupt 和状态查看

## 1. Subgraph 解决什么问题

Subgraph 就是：

```text
把一段 graph 封装起来，作为父 graph 的一个节点使用。
```

它常用于：

- 多智能体系统里的 specialist agent
- 把复杂 workflow 拆成独立模块
- 不同团队维护不同子流程
- 复用一段稳定的图逻辑

关键不是“嵌套更高级”，而是：

```text
父图只关心子图输入输出边界，子图内部怎么跑可以独立维护。
```

In [ ]:
import importlib.metadata

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from typing_extensions import TypedDict

print("langgraph", importlib.metadata.version("langgraph"))

## 2. 两种通信方式

官方文档里最重要的是这张判断表：

| 场景 | 做法 |
| --- | --- |
| 父图和子图 state schema 不同 | 在父图节点函数里调用 `subgraph.invoke(...)`，手动做输入/输出转换 |
| 父图和子图共享 state key | 把 compiled subgraph 直接传给 `add_node(...)` |

Java 类比：

```text
不同 DTO：需要 adapter / mapper。
相同接口：可以直接组合调用。
```

## 3. 模式一：不同 schema，用 wrapper node 调用子图

子图 state：

```text
bar, baz
```

父图 state：

```text
foo
```

两边没有共享 key，所以父图要写一个 `call_private_subgraph` 做转换。

In [ ]:
class PrivateSubgraphState(TypedDict):
    bar: str
    baz: str


def private_prepare(state: PrivateSubgraphState) -> dict:
    return {"baz": "baz"}


def private_finish(state: PrivateSubgraphState) -> dict:
    return {"bar": state["bar"] + state["baz"]}


private_subgraph = (
    StateGraph(PrivateSubgraphState)
    .add_node("private_prepare", private_prepare)
    .add_node("private_finish", private_finish)
    .add_edge(START, "private_prepare")
    .add_edge("private_prepare", "private_finish")
    .add_edge("private_finish", END)
    .compile()
)


class ParentOnlyState(TypedDict):
    foo: str


def parent_prefix(state: ParentOnlyState) -> dict:
    return {"foo": "hi " + state["foo"]}


def call_private_subgraph(state: ParentOnlyState) -> dict:
    subgraph_input = {"bar": state["foo"], "baz": ""}
    subgraph_output = private_subgraph.invoke(subgraph_input)
    return {"foo": subgraph_output["bar"]}


wrapper_parent_graph = (
    StateGraph(ParentOnlyState)
    .add_node("parent_prefix", parent_prefix)
    .add_node("call_private_subgraph", call_private_subgraph)
    .add_edge(START, "parent_prefix")
    .add_edge("parent_prefix", "call_private_subgraph")
    .add_edge("call_private_subgraph", END)
    .compile()
)

wrapper_parent_graph.invoke({"foo": "foo"})

## 4. 观察 wrapper 子图 stream

`subgraphs=True` 会让 stream 输出包含子图事件。

在 `version="v2"` 里，每个 stream part 有：

```python
part["ns"]
```

`ns == ()` 表示父图事件。

`ns != ()` 表示子图事件，里面会带子图节点名和运行 ID。

In [ ]:
for part in wrapper_parent_graph.stream(
    {"foo": "foo"},
    stream_mode="updates",
    subgraphs=True,
    version="v2",
):
    print("ns=", part["ns"], "data=", part["data"])

## 5. 模式二：共享 state key，直接 add_node 子图

这次父图和子图都共享 `foo`。

子图还有自己的内部字段 `bar`。

父图只看到最终写回来的 `foo`。

In [ ]:
class SharedSubgraphState(TypedDict):
    foo: str
    bar: str


def shared_prepare(state: SharedSubgraphState) -> dict:
    return {"bar": "bar"}


def shared_finish(state: SharedSubgraphState) -> dict:
    return {"foo": state["foo"] + state["bar"]}


shared_subgraph = (
    StateGraph(SharedSubgraphState)
    .add_node("shared_prepare", shared_prepare)
    .add_node("shared_finish", shared_finish)
    .add_edge(START, "shared_prepare")
    .add_edge("shared_prepare", "shared_finish")
    .add_edge("shared_finish", END)
    .compile()
)


class SharedParentState(TypedDict):
    foo: str


def shared_parent_prefix(state: SharedParentState) -> dict:
    return {"foo": "hi " + state["foo"]}


shared_parent_graph = (
    StateGraph(SharedParentState)
    .add_node("shared_parent_prefix", shared_parent_prefix)
    .add_node("shared_subgraph", shared_subgraph)
    .add_edge(START, "shared_parent_prefix")
    .add_edge("shared_parent_prefix", "shared_subgraph")
    .add_edge("shared_subgraph", END)
    .compile()
)

shared_parent_graph.invoke({"foo": "foo"})

In [ ]:
for part in shared_parent_graph.stream(
    {"foo": "foo"},
    stream_mode="updates",
    subgraphs=True,
    version="v2",
):
    print("ns=", part["ns"], "data=", part["data"])

## 6. 子图里的 interrupt 会向父图透传

子图里也可以调用 `interrupt()`。

只要父图有 checkpointer，暂停请求会回到父图调用方。

In [ ]:
class InterruptState(TypedDict):
    foo: str


def ask_inside_subgraph(state: InterruptState) -> dict:
    value = interrupt("子图需要一个人工输入值")
    return {"foo": state["foo"] + value}


interrupt_subgraph = (
    StateGraph(InterruptState)
    .add_node("ask_inside_subgraph", ask_inside_subgraph)
    .add_edge(START, "ask_inside_subgraph")
    .add_edge("ask_inside_subgraph", END)
    .compile()
)

interrupt_parent_graph = (
    StateGraph(InterruptState)
    .add_node("interrupt_child", interrupt_subgraph)
    .add_edge(START, "interrupt_child")
    .add_edge("interrupt_child", END)
    .compile(checkpointer=InMemorySaver())
)

interrupt_config = {"configurable": {"thread_id": "subgraph-interrupt-demo"}}

interrupt_first = interrupt_parent_graph.invoke({"foo": "base-"}, interrupt_config)
print("parent result:", interrupt_first)
print("interrupt value:", interrupt_first["__interrupt__"][0].value)

## 7. 查看暂停时的子图 state

`get_state(config, subgraphs=True)` 可以看到暂停时的子图 state。

这对排查嵌套工作流很重要。

In [ ]:
parent_snapshot = interrupt_parent_graph.get_state(interrupt_config, subgraphs=True)
subgraph_snapshot = parent_snapshot.tasks[0].state

print("parent next:", parent_snapshot.next)
print("subgraph values:", subgraph_snapshot.values)
print("subgraph next:", subgraph_snapshot.next)

interrupt_final = interrupt_parent_graph.invoke(Command(resume="ok"), interrupt_config)
print("final after resume:", interrupt_final)

## 8. 子图 checkpointer 模式

子图 compile 时可以控制持久化方式：

| 模式 | `checkpointer=` | 含义 |
| --- | --- | --- |
| per-invocation | `None` 默认 | 每次调用子图都从新状态开始，但单次调用内可继承父图 checkpointer |
| per-thread | `True` | 子图在同一个 thread 内跨调用积累状态 |
| stateless | `False` | 像普通函数一样运行，没有 checkpoint、interrupt、恢复能力 |

大多数 multi-agent subagent 场景默认用 per-invocation。

只有子图自己需要多轮记忆时，才考虑 `checkpointer=True`。

## 9. 和 Harness 多智能体的关系

| LangGraph Subgraph | Harness 风格多 agent |
| --- | --- |
| subgraph node | subagent / specialist worker |
| wrapper node | coordinator 的输入输出适配 |
| shared state key | 共享消息或共享任务状态 |
| private subgraph state | 子智能体私有上下文 |
| `subgraphs=True` | 观察子任务事件流 |
| subgraph checkpointer | 子智能体生命周期和记忆策略 |

关键原则：

```text
子图不是随便嵌套一层，而是给子流程定义清晰边界。
父图负责组合和综合，子图负责局部流程。
```

## 10. 本讲练习

请判断下面场景应该用哪种子图通信方式：

1. 父图 state 是 `{question, final_answer}`，研究子图 state 是 `{query, notes}`。
2. 父图和子图都围绕 `messages` 追加消息。
3. 子图内部有私有字段 `draft`，但最后只写回父图的 `answer`。
4. 子图只是一个可复用的校验流程，输入输出和父图完全一致。

参考答案：

1. wrapper node 调用子图，手动映射 state
2. compiled subgraph 直接 add_node
3. wrapper node 调用子图，隐藏私有字段
4. compiled subgraph 直接 add_node

## 11. 本讲小结

这一讲的核心：

```text
Subgraph = 可组合的局部 graph。
不同 schema 用 wrapper，共享 key 直接 add_node。
```

你现在应该能看懂：

- `subgraph.invoke(...)` 作为 wrapper node 的内部调用
- `builder.add_node("name", compiled_subgraph)`
- `stream(..., subgraphs=True, version="v2")`
- `part["ns"]` 如何区分父图和子图事件
- 子图 interrupt 如何透传到父图
- `get_state(config, subgraphs=True)` 如何查看子图状态
- `checkpointer=None / True / False` 的区别

下一讲可以继续学习 Deployment / Application Structure。